In [6]:
from sqlite3 import connect
import pandas as pd

In [91]:
conn = connect(':memory:')
df = pd.read_csv('daily_temp_data/bikes_table.csv')
df.to_sql(name='bikes', con=conn)

45275

In [196]:

sql_query = """

WITH bike_trip_cte AS (
SELECT
    is_standalone,
    bike_id,
    date,
    place_uid AS new_place,
    bike_lat AS new_lat,
    bike_lng AS new_lng,
    LAG(place_uid, 1) OVER (PARTITION BY bike_id ORDER BY date) AS last_place,
    LAG(bike_lng, 1) OVER (PARTITION BY bike_id ORDER BY date) AS last_lng,
    LAG(bike_lat, 1) OVER (PARTITION BY bike_id ORDER BY date) AS last_lat
FROM bikes
), new_cte AS (
SELECT
    is_standalone,
    bike_id,
    date,
    new_place,
    last_place,
    last_lat,
    last_lng
FROM bike_trip_cte
WHERE last_place != new_place

), trips AS (
SELECT
    is_standalone,
    bike_id,
    date,
    LAG(date, 1) OVER (PARTITION BY bike_id ORDER BY date) AS last_updated,
    new_place,
    last_place,
    last_lat,
    last_lng
FROM new_cte
)
SELECT
    is_standalone,
    bike_id,
    date,
    last_updated,
    last_lng,
    last_lat,
    ROUND(CAST((julianday(date) - julianday(last_updated))*24*60 AS REAL), 0) AS trip_length
FROM trips
WHERE last_updated IS NOT NULL
    AND ROUND(CAST((julianday(date) - julianday(last_updated))*24*60 AS REAL), 0) > 1
LIMIT 10

"""

qqry = """ SELECT * FROM bikes"""

pd.read_sql(sql_query, conn)

,is_standalone,bike_id,date,last_updated,last_lng,last_lat,trip_length
0,1,10354,2025-04-24 02:26:00,2025-04-24 02:21:00,13.439567,52.486498,5.0
1,1,10766,2025-04-24 02:27:00,2025-04-24 02:23:00,13.422710,52.488613,4.0
2,1,13219,2025-04-24 02:24:00,2025-04-24 02:21:00,13.529686,52.451618,3.0
3,1,13872,2025-04-24 02:27:00,2025-04-24 02:24:00,13.321799,52.501818,3.0
4,1,14311,2025-04-24 02:23:00,2025-04-24 02:21:00,13.197640,52.524333,2.0
5,1,14484,2025-04-24 02:29:00,2025-04-24 02:26:00,13.359082,52.481867,3.0
6,1,15995,2025-04-24 02:27:00,2025-04-24 02:24:00,13.379749,52.516302,3.0
7,1,16418,2025-04-24 02:28:00,2025-04-24 02:25:00,13.404200,52.522413,3.0
8,1,16828,2025-04-24 02:29:00,2025-04-24 02:26:00,13.444056,52.482951,3.0
9,1,17011,2025-04-24 02:24:00,2025-04-24 02:22:00,13.546920,52.453973,2.0
